# 08. Playground and Test LLM-Judge Calls

This notebook has two explicit modes. `playground` judges all test-split documents from the author group containing exploratory document 188; `test` judges the remaining reference-complete test documents and excludes that entire author group. The source split is unchanged. Results are separated by mode and fingerprint.

Assessment quality uses German prompt `0.5.0`, which includes the candidate-aware aggregate gold segment-analysis summary, teacher-educator feedback, reflection, rubric, and generated assessment. It must not infer segment-to-band alignments from aggregate counts. Feedback quality uses German prompt `0.3.0` and remains unchanged. The normal workflow rejudges assessment only; existing reference-aware feedback results remain valid.

Use playground results to calibrate and freeze the assessment judge before switching `JUDGE_MODE` to `test`. Do not include playground-author reflections in final test summaries. Provider calls require approval and are rate-limited; protected results include prompt, input, teacher-reference, and gold-summary provenance hashes.

In [1]:
import importlib
import json
import os
from collections import Counter
from pathlib import Path
from uuid import UUID

from dotenv import load_dotenv
import reflection_assessment_feedback.development_cohorts as cohort_module
from reflection_assessment_feedback.experiment_config import load_experiment_config
from reflection_assessment_feedback.prompt_registry import resolve_prompt

cohort_module = importlib.reload(cohort_module)
build_gold_segment_analysis_summary = cohort_module.build_gold_segment_analysis_summary
load_author_document_group = cohort_module.load_author_document_group
if not callable(build_gold_segment_analysis_summary):
    raise RuntimeError('Gold segment summary builder is unavailable.')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'prebi_v1':
    raise RuntimeError('Run this notebook from the prebi_v1 project root.')
load_dotenv(PROJECT_ROOT / '.env', override=False)

protected_root_value = os.getenv('PREBI_DATA_ROOT')
if not protected_root_value:
    raise RuntimeError('Set PREBI_DATA_ROOT to approved protected storage.')
PROTECTED_ROOT = Path(protected_root_value).expanduser().resolve()
if PROTECTED_ROOT == PROJECT_ROOT or PROJECT_ROOT in PROTECTED_ROOT.parents:
    raise ValueError('PREBI_DATA_ROOT must be outside the project source tree.')

EXPERIMENT_CONFIG = load_experiment_config()
JUDGE_CONFIG = EXPERIMENT_CONFIG['llm_judge']
DATASET_CONFIG = EXPERIMENT_CONFIG['dataset']
SPLIT_PATH = (
    PROJECT_ROOT
    / EXPERIMENT_CONFIG['paths']['provisional_split_directory']
    / f"provisional_{DATASET_CONFIG['prediction_split']}_modeling_wide.csv"
)
COHORT_SOURCES = [
    {'cohort_id': 'R1-development', 'expected_documents': 6, 'file_prefix': 'r1'},
    {
        'cohort_id': DATASET_CONFIG['generation_cohort'],
        'expected_documents': DATASET_CONFIG['expected_generation_document_count'],
        'file_prefix': DATASET_CONFIG['generation_cohort'].lower().replace('_', '-'),
    },
]
SPLIT_MANIFEST_PATH = (
    PROJECT_ROOT
    / EXPERIMENT_CONFIG['paths']['provisional_split_directory']
    / 'provisional_split_manifest.csv'
)
PLAYGROUND_AUTHOR_DOCUMENTS = load_author_document_group(
    SPLIT_MANIFEST_PATH,
    DATASET_CONFIG['exploratory_document_id'],
)
PLAYGROUND_DOCUMENT_IDS = set(PLAYGROUND_AUTHOR_DOCUMENTS['document_id'].astype(str))

JUDGE_PROVIDER = JUDGE_CONFIG['provider'].strip()
JUDGE_MODEL_NAME = JUDGE_CONFIG['model_name'].strip()
if not JUDGE_PROVIDER or not JUDGE_MODEL_NAME:
    raise RuntimeError('Configure an approved judge provider and model in [llm_judge].')
if JUDGE_PROVIDER != 'openai':
    raise ValueError('This notebook currently supports only the configured OpenAI adapter.')

ASSESSMENT_PROMPT = resolve_prompt(
    EXPERIMENT_CONFIG['prompts']['assessment_quality_llm_id'],
    EXPERIMENT_CONFIG['prompts']['assessment_quality_llm_version'],
)
FEEDBACK_PROMPT = resolve_prompt(
    EXPERIMENT_CONFIG['prompts']['feedback_quality_llm_id'],
    EXPERIMENT_CONFIG['prompts']['feedback_quality_llm_version'],
)
if any(prompt.status != 'development_draft' for prompt in (ASSESSMENT_PROMPT, FEEDBACK_PROMPT)):
    raise ValueError('This pilot requires the registered development-draft judge prompts.')

PACKET_DIRECTORY = PROTECTED_ROOT / 'rating-packets' / 'development'
for cohort_source in COHORT_SOURCES:
    prefix = cohort_source['file_prefix']
    cohort_source['packet_path'] = PACKET_DIRECTORY / f'{prefix}-rating-packets.json'
    cohort_source['key_path'] = PACKET_DIRECTORY / f'{prefix}-condition-key.json'

JUDGE_MODE = 'playground'
if JUDGE_MODE not in {'playground', 'test'}:
    raise ValueError('JUDGE_MODE must be playground or test.')
TASKS_TO_JUDGE = {'assessment_quality'}
APPROVE_EXTERNAL_PROCESSING = True
PILOT_ONLY = False
APPROVE_FULL_BATCH = True
ENABLE_LANGSMITH_TRACING = bool(JUDGE_CONFIG['enable_langsmith_tracing'])
print(
    f'Judge mode={JUDGE_MODE}; author-group documents={len(PLAYGROUND_DOCUMENT_IDS)}; '
    f'tasks={sorted(TASKS_TO_JUDGE)}.'
)

Judge mode=playground; author-group documents=4; tasks=['assessment_quality'].


## 1. Validate blinded packet inputs

The packet bundle contains protected reflection and generated-output text. The separate condition key is used only to link each packet to its source artifact hash; condition, document, run, and model identifiers are not passed to the judge.

In [2]:
quality_packets = []
key_by_packet = {}
teacher_reference_by_document = {}
teacher_reference_reflection_by_document = {}
gold_analysis_summary_by_document = {}
cohort_document_ids = {}
for cohort_source in COHORT_SOURCES:
    if not cohort_source['packet_path'].is_file() or not cohort_source['key_path'].is_file():
        raise FileNotFoundError(f"Missing packet export for {cohort_source['cohort_id']}.")
    packet_bundle = json.loads(cohort_source['packet_path'].read_text(encoding='utf-8'))
    condition_key = json.loads(cohort_source['key_path'].read_text(encoding='utf-8'))
    cohort_id = cohort_source['cohort_id']
    if condition_key.get('cohort') != cohort_id:
        raise ValueError('Packet-key cohort ID differs from its configured source.')
    cohort_key = {row['packet_id']: row for row in condition_key.get('records', [])}
    cohort_packets = packet_bundle.get('packets', [])
    cohort_quality = [
        packet for packet in cohort_packets
        if packet.get('task') in {'assessment_quality', 'feedback_quality'}
    ]
    cohort_quality_ids = {packet.get('packet_id') for packet in cohort_quality}
    if not cohort_quality_ids.issubset(cohort_key):
        raise ValueError('A quality packet is missing its protected source-key record.')
    if cohort_quality_ids.intersection(key_by_packet):
        raise ValueError('Packet IDs must be unique across cohorts.')

    references = {}
    reference_reflections = {}
    for reference_packet in cohort_packets:
        if reference_packet.get('task') != 'feedback_implied_score':
            continue
        reference_key = cohort_key.get(reference_packet.get('packet_id'))
        if reference_key is None:
            raise ValueError('A teacher-reference packet is missing its protected key record.')
        document_id = str(reference_key['document_id'])
        reference_feedback = reference_packet.get('human_feedback')
        if not isinstance(reference_feedback, str) or not reference_feedback.strip():
            raise ValueError('A teacher-reference packet has no human-written feedback.')
        identity = (cohort_id, document_id)
        if identity in references:
            raise ValueError('More than one teacher reference was found for a cohort document.')
        references[identity] = reference_feedback
        reference_reflections[identity] = reference_packet.get('displayed_reflection_sha256')

    document_ids = {str(cohort_key[packet_id]['document_id']) for packet_id in cohort_quality_ids}
    if len(document_ids) != cohort_source['expected_documents']:
        raise ValueError('Quality packets do not cover the configured cohort size.')
    if {identity[1] for identity in references} != document_ids:
        raise ValueError('Teacher references do not cover exactly the quality documents.')

    reflection_by_document = {}
    for packet in cohort_quality:
        packet_id = packet['packet_id']
        key_record = {**cohort_key[packet_id], 'cohort_id': cohort_id}
        document_id = str(key_record['document_id'])
        identity = (cohort_id, document_id)
        reflection_hash = packet.get('displayed_reflection_sha256')
        previous_hash = reflection_by_document.setdefault(document_id, reflection_hash)
        if previous_hash != reflection_hash:
            raise ValueError('Quality packets disagree on a reflection snapshot.')
        if reference_reflections[identity] != reflection_hash:
            raise ValueError('Teacher reference is attached to a different reflection.')
        if packet['task'] == 'assessment_quality':
            summary = build_gold_segment_analysis_summary(SPLIT_PATH, document_id)
            if summary['segment_count'] != len(packet.get('reflection', {}).get('segments', [])):
                raise ValueError('Gold summary segment count differs from the judged reflection.')
            gold_analysis_summary_by_document[identity] = summary
        key_by_packet[packet_id] = key_record
        quality_packets.append(packet)
    teacher_reference_by_document.update(references)
    teacher_reference_reflection_by_document.update(reference_reflections)
    cohort_document_ids[cohort_id] = document_ids

ALL_DOCUMENT_IDS = set().union(*cohort_document_ids.values())
if len(ALL_DOCUMENT_IDS) != 22:
    raise ValueError('Expected 22 reference-complete documents across both packet cohorts.')
if not PLAYGROUND_DOCUMENT_IDS.issubset(ALL_DOCUMENT_IDS):
    raise ValueError('Some author-playground documents lack a complete packet/reference set.')
TEST_DOCUMENT_IDS = ALL_DOCUMENT_IDS - PLAYGROUND_DOCUMENT_IDS
if not TEST_DOCUMENT_IDS:
    raise ValueError('The final test cohort is empty after excluding the playground author.')
JUDGE_DOCUMENT_IDS = (
    PLAYGROUND_DOCUMENT_IDS if JUDGE_MODE == 'playground' else TEST_DOCUMENT_IDS
)
if len(quality_packets) != len(ALL_DOCUMENT_IDS) * 6:
    raise ValueError('Expected six quality packets per document across both source cohorts.')

def judge_packet_for(packet):
    key_record = key_by_packet[packet['packet_id']]
    identity = (key_record['cohort_id'], str(key_record['document_id']))
    payload = {
        **packet,
        'reference_teacher_feedback': teacher_reference_by_document[identity],
    }
    if packet['task'] == 'assessment_quality':
        payload['gold_analysis_summary'] = gold_analysis_summary_by_document[identity]
    return payload

for packet in quality_packets:
    packet_id = packet['packet_id']
    if packet.get('condition_blinded') is not True:
        raise ValueError('An unblinded quality packet was found.')
    if {'condition', 'document_id', 'run_id', 'model_name'}.intersection(packet):
        raise ValueError('A quality packet contains generation provenance fields.')
    if not key_by_packet[packet_id].get('source_artifact_sha256'):
        raise ValueError('A packet source artifact hash is missing from the protected key.')
print(
    f'Validated {len(quality_packets)} blinded quality packets, '
    f'{len(teacher_reference_by_document)} references, {len(gold_analysis_summary_by_document)} gold summaries; '
    f'mode={JUDGE_MODE}, documents={len(JUDGE_DOCUMENT_IDS)}; text withheld.'
)

Validated 132 blinded quality packets, 22 references, 22 gold summaries; mode=playground, documents=4; text withheld.


## 2. Run the selected phase

Cell 2 selects only assessment-quality calls. In `playground` mode this is 12 packets (4 reflections × 3 conditions); after calibration and freezing the prompt, change `JUDGE_MODE` to `test` to select 54 packets (18 × 3). The two modes use separate result fingerprints. Do not include playground results in final test summaries.

In [12]:
if not APPROVE_EXTERNAL_PROCESSING:
    raise RuntimeError('Set APPROVE_EXTERNAL_PROCESSING = True only after provider approval.')
if not PILOT_ONLY and not APPROVE_FULL_BATCH:
    raise RuntimeError('Full-batch judging requires APPROVE_FULL_BATCH = True.')
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Configure OPENAI_API_KEY in the approved environment.')
if ENABLE_LANGSMITH_TRACING:
    if not os.getenv('LANGSMITH_API_KEY') or not os.getenv('LANGSMITH_PROJECT'):
        raise RuntimeError('Tracing requires approved LANGSMITH_API_KEY and LANGSMITH_PROJECT settings.')
    if os.getenv('LANGSMITH_ENDPOINT', 'https://eu.api.smith.langchain.com').rstrip('/') != 'https://eu.api.smith.langchain.com':
        raise ValueError('Judge tracing is restricted to the approved EU endpoint.')

import hashlib
from langchain_openai import ChatOpenAI
from reflection_assessment_feedback.experiment_config import experiment_config_sha256
from reflection_assessment_feedback.llm_judge import (
    judge_input_sha256,
    run_quality_judge,
)
from reflection_assessment_feedback.rate_limit import reserve_request_slot

prompt_fingerprint = {
    'provider': JUDGE_PROVIDER,
    'model': JUDGE_MODEL_NAME,
    'assessment_prompt_sha256': ASSESSMENT_PROMPT.sha256,
    'feedback_prompt_sha256': FEEDBACK_PROMPT.sha256,
    'experiment_config_sha256': experiment_config_sha256(),
    'judge_mode': JUDGE_MODE,
    'document_ids': sorted(JUDGE_DOCUMENT_IDS),
}
JUDGE_FINGERPRINT = hashlib.sha256(
    json.dumps(prompt_fingerprint, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
RESULT_DIRECTORY = PROTECTED_ROOT / 'llm-judge' / 'development' / JUDGE_FINGERPRINT
RESULT_DIRECTORY.mkdir(parents=True, exist_ok=True, mode=0o700)
os.chmod(RESULT_DIRECTORY, 0o700)
if not RESULT_DIRECTORY.resolve().is_relative_to(PROTECTED_ROOT):
    raise ValueError('Judge result storage must resolve within PREBI_DATA_ROOT.')
RATE_LIMIT_STATE = PROTECTED_ROOT / '.rate_limits' / 'openai-judge.state'
judge_model = ChatOpenAI(model=JUDGE_MODEL_NAME)

def result_path_for(packet_id):
    return RESULT_DIRECTORY / f'judge-{UUID(packet_id).hex}.json'

def write_result_atomically(path, content):
    import tempfile
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode='w', encoding='utf-8', dir=RESULT_DIRECTORY,
            prefix='.judge-', suffix='.tmp', delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)
            json.dump(content, temporary_file, ensure_ascii=False, indent=2)
            temporary_file.write('\n')
        os.chmod(temporary_path, 0o600)
        if path.exists():
            raise FileExistsError('Refusing to overwrite an existing judge result.')
        os.replace(temporary_path, path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

selected_packets_for_mode = [
    packet for packet in quality_packets
    if packet['task'] in TASKS_TO_JUDGE
    and str(key_by_packet[packet['packet_id']]['document_id']) in JUDGE_DOCUMENT_IDS
]
if PILOT_ONLY:
    selected_packets = []
    seen_tasks = set()
    for packet in selected_packets_for_mode:
        if packet['task'] not in seen_tasks:
            selected_packets.append(packet)
            seen_tasks.add(packet['task'])
else:
    selected_packets = selected_packets_for_mode

new_results = 0
reused_results = 0
for packet in selected_packets_for_mode:
    saved_path = result_path_for(packet['packet_id'])
    if not saved_path.exists():
        continue
    saved = json.loads(saved_path.read_text(encoding='utf-8'))
    judge_packet = judge_packet_for(packet)
    reference_sha256 = hashlib.sha256(
        judge_packet['reference_teacher_feedback'].encode('utf-8')
    ).hexdigest()
    expected_gold_summary_sha256 = None
    if packet['task'] == 'assessment_quality':
        expected_gold_summary_sha256 = hashlib.sha256(
            json.dumps(
                judge_packet['gold_analysis_summary'],
                ensure_ascii=False,
                sort_keys=True,
                separators=(',', ':'),
            ).encode('utf-8')
        ).hexdigest()
    if (
        saved.get('packet_id') != packet['packet_id']
        or saved.get('task') != packet['task']
        or saved.get('evaluation_mode') != JUDGE_MODE
        or saved.get('input_packet_sha256') != judge_input_sha256(judge_packet)
        or saved.get('teacher_reference_sha256') != reference_sha256
        or saved.get('gold_analysis_summary_sha256') != expected_gold_summary_sha256
        or saved.get('judge_provider') != JUDGE_PROVIDER
        or saved.get('judge_model') != JUDGE_MODEL_NAME
    ):
        raise ValueError('An existing result conflicts with current inputs, mode, or judge configuration.')
    reused_results += 1

for packet in selected_packets:
    judge_packet = judge_packet_for(packet)
    reserve_request_slot(RATE_LIMIT_STATE, JUDGE_CONFIG['minimum_request_interval_seconds'])
    result = run_quality_judge(
        model=judge_model,
        packet=judge_packet,
        provider=JUDGE_PROVIDER,
        model_name=JUDGE_MODEL_NAME,
        enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
    )
    result['evaluation_mode'] = JUDGE_MODE
    result['evaluation_document_id'] = key_by_packet[packet['packet_id']]['document_id']
    result['evaluation_cohort_id'] = key_by_packet[packet['packet_id']]['cohort_id']
    result['source_artifact_sha256'] = key_by_packet[packet['packet_id']]['source_artifact_sha256']
    result['teacher_reference_sha256'] = hashlib.sha256(
        judge_packet['reference_teacher_feedback'].encode('utf-8')
    ).hexdigest()
    if packet['task'] == 'assessment_quality':
        result['gold_analysis_summary_sha256'] = hashlib.sha256(
            json.dumps(
                judge_packet['gold_analysis_summary'],
                ensure_ascii=False,
                sort_keys=True,
                separators=(',', ':'),
            ).encode('utf-8')
        ).hexdigest()
    result['experiment_config_sha256'] = experiment_config_sha256()
    write_result_atomically(result_path_for(packet['packet_id']), result)
    new_results += 1

expected_mode_results = len(JUDGE_DOCUMENT_IDS) * 3 * len(TASKS_TO_JUDGE)
if new_results + reused_results != expected_mode_results:
    raise ValueError('Judge results do not cover every selected document/condition/task in this mode.')
print(
    f"Mode={JUDGE_MODE}; judge calls={new_results}; reused={reused_results}; "
    f"documents={len(JUDGE_DOCUMENT_IDS)}; tasks={sorted(TASKS_TO_JUDGE)}."
)
print(f'Protected result directory: {RESULT_DIRECTORY}')

Mode=playground; judge calls=12; reused=0; documents=4; tasks=['assessment_quality'].
Protected result directory: /Users/quynguyen/Documents/CodingProjects/prebi_data/llm-judge/development/2e2713ca24c6d523bae1128661aa36bc6e6803e7d6fbe54a8bd8259eff77350c


## 3. Verify saved results

This check reports counts only. Result files contain judge rationales and feedback-span comments; keep them protected. Human agreement analysis is a separate step and must join results through the restricted condition key only after judge outputs are finalized.

In [13]:
saved_results = {}
packet_by_id = {packet['packet_id']: packet for packet in quality_packets}
for path in RESULT_DIRECTORY.glob('judge-*.json'):
    record = json.loads(path.read_text(encoding='utf-8'))
    packet_id = record.get('packet_id')
    if packet_id not in packet_by_id:
        raise ValueError('A saved judge result does not match a selected quality packet.')
    packet = packet_by_id[packet_id]
    key_record = key_by_packet[packet_id]
    if packet['task'] not in TASKS_TO_JUDGE:
        raise ValueError('A result was saved for a task not selected in this run.')
    if str(key_record['document_id']) not in JUDGE_DOCUMENT_IDS:
        raise ValueError('A result was saved for a document outside this evaluation mode.')
    if record.get('evaluation_mode') != JUDGE_MODE:
        raise ValueError('A result evaluation mode differs from the selected mode.')
    judge_packet = judge_packet_for(packet)
    reference_sha256 = hashlib.sha256(
        judge_packet['reference_teacher_feedback'].encode('utf-8')
    ).hexdigest()
    if record.get('input_packet_sha256') != judge_input_sha256(judge_packet):
        raise ValueError('A saved judge result has a mismatched input fingerprint.')
    if record.get('teacher_reference_sha256') != reference_sha256:
        raise ValueError('A saved result has a mismatched teacher-reference fingerprint.')
    if packet['task'] == 'assessment_quality':
        summary_sha256 = hashlib.sha256(
            json.dumps(
                judge_packet['gold_analysis_summary'],
                ensure_ascii=False,
                sort_keys=True,
                separators=(',', ':'),
            ).encode('utf-8')
        ).hexdigest()
        if record.get('gold_analysis_summary_sha256') != summary_sha256:
            raise ValueError('A saved assessment result has a mismatched gold-summary fingerprint.')
    if record.get('record_type') != 'llm_judge_quality_rating':
        raise ValueError('A saved result has an unexpected record type.')
    if record.get('judge_provider') != JUDGE_PROVIDER or record.get('judge_model') != JUDGE_MODEL_NAME:
        raise ValueError('A saved result has a mismatched judge identity.')
    expected_prompt = ASSESSMENT_PROMPT if packet['task'] == 'assessment_quality' else FEEDBACK_PROMPT
    if record.get('prompt_id') != expected_prompt.artifact_id or record.get('prompt_version') != expected_prompt.version:
        raise ValueError('A saved result has mismatched judge prompt provenance.')
    saved_results[packet_id] = record

saved_counts = Counter(record['task'] for record in saved_results.values())
expected_mode_results = len(JUDGE_DOCUMENT_IDS) * 3 * len(TASKS_TO_JUDGE)
if len(saved_results) != expected_mode_results or set(saved_counts) != TASKS_TO_JUDGE:
    raise ValueError('Saved judge results do not fully cover the selected mode.')
print(
    f"Validated mode={JUDGE_MODE}: {dict(saved_counts)}, "
    f"documents={len(JUDGE_DOCUMENT_IDS)}. Result text withheld."
)

Validated mode=playground: {'assessment_quality': 12}, documents=4. Result text withheld.


In [3]:
import hashlib
import json
import os
import tempfile
from datetime import datetime, timezone
from uuid import UUID

import pandas as pd
from IPython.display import display
from langchain_openai import ChatOpenAI
from langchain_core.exceptions import OutputParserException
from pydantic import ValidationError

from reflection_assessment_feedback.experiment_config import experiment_config_sha256
from reflection_assessment_feedback.llm_judge import judge_input_sha256, run_quality_judge
from reflection_assessment_feedback.rate_limit import reserve_request_slot

if not APPROVE_EXTERNAL_PROCESSING or not APPROVE_FULL_BATCH:
    raise RuntimeError('Explicit external-processing and full-batch approval are required.')
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Configure OPENAI_API_KEY in the approved environment.')
if ENABLE_LANGSMITH_TRACING:
    if not os.getenv('LANGSMITH_API_KEY') or not os.getenv('LANGSMITH_PROJECT'):
        raise RuntimeError('Configure approved LangSmith credentials and project.')
    if os.getenv('LANGSMITH_ENDPOINT', 'https://eu.api.smith.langchain.com').rstrip('/') != 'https://eu.api.smith.langchain.com':
        raise ValueError('Tracing requires the approved EU endpoint.')
full_judge_tasks = {'assessment_quality', 'feedback_quality'}
full_judge_model = ChatOpenAI(model=JUDGE_MODEL_NAME)
full_judge_records = []
full_judge_phase_counts = []
full_judge_directories = {}
full_judge_config_hash = experiment_config_sha256()

for full_mode, full_documents in (
    ('playground', PLAYGROUND_DOCUMENT_IDS), ('test', TEST_DOCUMENT_IDS),
):
    full_fingerprint_payload = {
        'provider': JUDGE_PROVIDER, 'model': JUDGE_MODEL_NAME,
        'assessment_prompt_sha256': ASSESSMENT_PROMPT.sha256,
        'feedback_prompt_sha256': FEEDBACK_PROMPT.sha256,
        'experiment_config_sha256': full_judge_config_hash,
        'judge_mode': full_mode, 'document_ids': sorted(full_documents),
    }
    full_fingerprint = hashlib.sha256(json.dumps(
        full_fingerprint_payload, sort_keys=True, separators=(',', ':'),
    ).encode()).hexdigest()
    full_directory = PROTECTED_ROOT / 'llm-judge' / 'development' / full_fingerprint
    if not full_directory.resolve().is_relative_to(PROTECTED_ROOT):
        raise ValueError('Result directory must remain in protected storage.')
    full_directory.mkdir(parents=True, exist_ok=True, mode=0o700)
    full_directory.chmod(0o700)
    full_judge_directories[full_mode] = full_directory
    full_packets = [
        packet for packet in quality_packets
        if packet['task'] in full_judge_tasks
        and str(key_by_packet[packet['packet_id']]['document_id']) in full_documents
    ]
    if len(full_packets) != len(full_documents) * 6:
        raise ValueError('Incomplete document/condition/task packet coverage.')
    full_new = 0
    full_reused = 0
    print(f'OpenAI judge phase {full_mode}: {len(full_documents)} documents, {len(full_packets)} packets.', flush=True)
    for full_number, full_packet in enumerate(full_packets, start=1):
        full_key = key_by_packet[full_packet['packet_id']]
        full_payload = judge_packet_for(full_packet)
        full_prompt = ASSESSMENT_PROMPT if full_packet['task'] == 'assessment_quality' else FEEDBACK_PROMPT
        full_gold_hash = None
        if full_packet['task'] == 'assessment_quality':
            full_gold_hash = hashlib.sha256(json.dumps(
                full_payload['gold_analysis_summary'], ensure_ascii=False,
                sort_keys=True, separators=(',', ':'),
            ).encode()).hexdigest()
        full_expected = {
            'record_type': 'llm_judge_quality_rating',
            'packet_id': full_packet['packet_id'], 'task': full_packet['task'],
            'evaluation_mode': full_mode,
            'evaluation_document_id': full_key['document_id'],
            'evaluation_cohort_id': full_key['cohort_id'],
            'input_packet_sha256': judge_input_sha256(full_payload),
            'source_artifact_sha256': full_key['source_artifact_sha256'],
            'teacher_reference_sha256': hashlib.sha256(full_payload['reference_teacher_feedback'].encode()).hexdigest(),
            'judge_provider': JUDGE_PROVIDER, 'judge_model': JUDGE_MODEL_NAME,
            'prompt_id': full_prompt.artifact_id, 'prompt_version': full_prompt.version,
            'prompt_artifact_sha256': full_prompt.sha256,
            'experiment_config_sha256': full_judge_config_hash,
        }
        full_path = full_directory / f"judge-{UUID(full_packet['packet_id']).hex}.json"
        if full_path.exists():
            full_result = json.loads(full_path.read_text(encoding='utf-8'))
            if any(full_result.get(key) != value for key, value in full_expected.items()):
                raise ValueError('Cached judge result conflicts with current provenance.')
            if full_result.get('gold_analysis_summary_sha256') != full_gold_hash:
                raise ValueError('Cached judge result has different gold-summary provenance.')
            full_reused += 1
        else:
            for full_attempt in range(1, 4):
                reserve_request_slot(
                    PROTECTED_ROOT / '.rate_limits' / 'openai-judge.state',
                    JUDGE_CONFIG['minimum_request_interval_seconds'],
                )
                try:
                    full_result = run_quality_judge(
                        model=full_judge_model, packet=full_payload,
                        provider=JUDGE_PROVIDER, model_name=JUDGE_MODEL_NAME,
                        enable_langsmith_tracing=ENABLE_LANGSMITH_TRACING,
                    )
                    break
                except (OutputParserException, ValidationError):
                    if full_attempt == 3:
                        raise
                    print(f'{full_mode} packet {full_number}: retrying invalid structured output.', flush=True)
            full_result.update(full_expected)
            if full_gold_hash is not None:
                full_result['gold_analysis_summary_sha256'] = full_gold_hash
            full_temporary_path = None
            try:
                with tempfile.NamedTemporaryFile(
                    mode='w', encoding='utf-8', dir=full_directory, suffix='.tmp', delete=False,
                ) as full_file:
                    full_temporary_path = Path(full_file.name)
                    json.dump(full_result, full_file, ensure_ascii=False, indent=2)
                full_temporary_path.chmod(0o600)
                if full_path.exists():
                    raise FileExistsError('Refusing to overwrite a judge result.')
                os.replace(full_temporary_path, full_path)
            finally:
                if full_temporary_path is not None:
                    full_temporary_path.unlink(missing_ok=True)
            full_new += 1
        full_judge_records.append(full_result)
        print(f'{full_mode}: validated {full_number}/{len(full_packets)} (new={full_new}, reused={full_reused}).', flush=True)
    full_judge_phase_counts.append({
        'mode': full_mode, 'documents': len(full_documents),
        'new_results': full_new, 'reused_results': full_reused,
    })

full_rating_rows = []
for record in full_judge_records:
    key = key_by_packet[record['packet_id']]
    for rating in record['result']['ratings']:
        full_rating_rows.append({
            'mode': record['evaluation_mode'], 'document_id': str(key['document_id']),
            'condition': key['condition'], 'task': record['task'],
            'criterion_id': rating['criterion_id'], 'score': rating['score'],
            'unable_to_judge': rating['unable_to_judge'],
        })
full_rating_frame = pd.DataFrame(full_rating_rows)
full_judge_comparison = full_rating_frame.groupby(
    ['mode', 'task', 'criterion_id', 'condition'], sort=True,
).agg(mean_score=('score', 'mean'), valid_ratings=('score', 'count'),
      total_ratings=('score', 'size'), unable_to_judge=('unable_to_judge', 'sum')).reset_index()
print('G1/G2/G3 comparison by criterion; playground is separate from final-test results:')
display(full_judge_comparison.round(4))
display(pd.DataFrame(full_judge_phase_counts))
full_summary_path = PROTECTED_ROOT / 'llm-judge' / 'development' / (
    'whole-test-comparison-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json'
)
full_descriptor = os.open(full_summary_path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
with os.fdopen(full_descriptor, 'w') as full_file:
    json.dump({
        'provider': JUDGE_PROVIDER, 'model': JUDGE_MODEL_NAME,
        'assessment_prompt_version': ASSESSMENT_PROMPT.version,
        'feedback_prompt_version': FEEDBACK_PROMPT.version,
        'phase_counts': full_judge_phase_counts,
        'comparison': full_judge_comparison.to_dict(orient='records'),
    }, full_file, indent=2)
print('Completed all 22 reference-complete test documents; two test documents have no complete packet/reference set. '
      'Results and aggregate comparison saved in protected storage.')

OpenAI judge phase playground: 4 documents, 24 packets.
playground: validated 1/24 (new=0, reused=1).
playground: validated 2/24 (new=1, reused=1).
playground: validated 3/24 (new=1, reused=2).
playground: validated 4/24 (new=2, reused=2).
playground: validated 5/24 (new=2, reused=3).
playground: validated 6/24 (new=3, reused=3).
playground: validated 7/24 (new=3, reused=4).
playground: validated 8/24 (new=4, reused=4).
playground: validated 9/24 (new=4, reused=5).
playground: validated 10/24 (new=5, reused=5).
playground: validated 11/24 (new=5, reused=6).
playground: validated 12/24 (new=6, reused=6).
playground: validated 13/24 (new=6, reused=7).
playground: validated 14/24 (new=7, reused=7).
playground: validated 15/24 (new=7, reused=8).
playground: validated 16/24 (new=8, reused=8).
playground: validated 17/24 (new=8, reused=9).
playground: validated 18/24 (new=9, reused=9).
playground: validated 19/24 (new=9, reused=10).
playground: validated 20/24 (new=10, reused=10).
playground

,mode,task,criterion_id,condition,mean_score,valid_ratings,total_ratings,unable_to_judge
0,playground,assessment_quality,evidence_support,G1,2.7500,4,4,0
1,playground,assessment_quality,evidence_support,G2,2.2500,4,4,0
2,playground,assessment_quality,evidence_support,G3,2.7500,4,4,0
3,playground,assessment_quality,justification_quality,G1,2.0000,4,4,0
4,playground,assessment_quality,justification_quality,G2,2.0000,4,4,0
5,playground,assessment_quality,justification_quality,G3,2.0000,4,4,0
6,playground,assessment_quality,score_rubric_fit,G1,2.2500,4,4,0
7,playground,assessment_quality,score_rubric_fit,G2,2.2500,4,4,0
8,playground,assessment_quality,score_rubric_fit,G3,2.2500,4,4,0
9,playground,feedback_quality,correctness,G1,2.7500,4,4,0


,mode,documents,new_results,reused_results
0,playground,4,12,12
1,test,18,108,0


Completed all 22 reference-complete test documents; two test documents have no complete packet/reference set. Results and aggregate comparison saved in protected storage.
